# Retrieval Debugger

Type a question → see which image crops the embedding model would retrieve, ranked by cosine similarity.

Use this to understand and tune retrieval: change the question, `top_k`, or `crop_label` filter and re-run just the **Query** cell.

> **Setup:** run the *Imports*, *Config*, *Load model*, and *Open index* cells once.  
> **Exploration:** only re-run the **Query** cell each time you change the question.

## Imports

In [ ]:
import os
import json
from pathlib import Path
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import plotly.graph_objects as go
from sklearn.decomposition import PCA

import torch
import chromadb
from chromadb.config import Settings
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration
from qwen_vl_utils import process_vision_info

from colette.kvstore import ImageStorageFactory

# Colour scheme for crop labels — used across all visualisations
LABEL_COLORS = {
    'text':     '#4C72B0',
    'figure':   '#55A868',
    'table':    '#C44E52',
    'overview': '#8172B2',
    'unknown':  '#999999',
}

## Config

Paths and parameters are read from the app's `config.json` — override any value below if needed.

In [ ]:
colette_root = Path(os.path.abspath(os.path.join(os.getcwd(), '..')))

# ── User configuration ───────────────────────────────────────────────────────
app_dir = colette_root / 'app_colette'    # ← directory passed to service_create
# ─────────────────────────────────────────────────────────────────────────────

with open(app_dir / 'config.json') as f:
    cfg = json.load(f)

rag_cfg          = cfg['parameters']['input']['rag']
embedding_model  = rag_cfg['embedding_model']          # e.g. Alibaba-NLP/gme-Qwen2-VL-2B-Instruct
models_repo      = Path(cfg['app']['models_repository'])

# ── Overrides ───────────────────────────────────────────────────────────────
GPU_ID  = 0     # GPU to load the embedding model on
TOP_K   = 8     # how many crops to retrieve
# ────────────────────────────────────────────────────────────────────────────

print(f'Embedding model : {embedding_model}')
print(f'Models repo     : {models_repo}')
print(f'GPU             : {GPU_ID}')
print(f'Top-K           : {TOP_K}')

## Load embedding model

Run once — takes ~30–60 s the first time (model download) or ~10 s from cache.

In [ ]:
device = f'cuda:{GPU_ID}'

print(f'Loading processor from {embedding_model} ...')
processor = AutoProcessor.from_pretrained(
    embedding_model,
    use_fast=False,          # keep Qwen2-VL processor stable
    min_pixels=1 * 28 * 28,
    max_pixels=2560 * 28 * 28,
    cache_dir=str(models_repo),
)
processor.tokenizer.padding_side = 'right'

print(f'Loading model ...')
embed_model = (
    Qwen2VLForConditionalGeneration.from_pretrained(
        embedding_model,
        torch_dtype=torch.bfloat16,
        cache_dir=str(models_repo),
    )
    .to(device)
    .eval()
)

print('Model loaded.')


def embed_text(query: str) -> list[float]:
    """Encode a text query the same way rag_img.py does for retrieval."""
    message = [{'role': 'user', 'content': [{'type': 'text', 'text': query}]}]
    text = (
        processor.apply_chat_template(message, tokenize=False, add_generation_prompt=True)
        + '<|endoftext|>'
    )
    image_inputs, video_inputs = process_vision_info([message])  # None, None for text-only
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        truncation=True,
        return_tensors='pt',
    ).to(device)
    with torch.no_grad():
        output = embed_model(
            **inputs,
            use_cache=False,
            return_dict=True,
            output_hidden_states=True,
        )
        # Qwen2VLForConditionalGeneration is a causal LM: hidden states live in
        # output.hidden_states (tuple), not output.last_hidden_state.
        if hasattr(output, 'last_hidden_state') and output.last_hidden_state is not None:
            hidden = output.last_hidden_state
        else:
            hidden = output.hidden_states[-1]   # (1, seq_len, hidden_dim)
        vec = hidden[:, -1]                     # last-token pooling
        vec = torch.nn.functional.normalize(vec, p=2, dim=-1)
    return vec.squeeze().tolist()

## Open ChromaDB index and kvstore

In [ ]:
index_path   = app_dir / 'mm_index'
kvstore_path = app_dir / 'kvstore.db'

chroma_client = chromadb.Client(
    Settings(
        is_persistent=True,
        persist_directory=str(index_path),
        anonymized_telemetry=False,
    )
)
# Open without embedding_function — we provide embeddings ourselves
collection = chroma_client.get_collection(name='mm_db')
kvstore    = ImageStorageFactory.create_storage('hdf5', kvstore_path, mode='r')

export_root = colette_root / 'tmp' / 'retrieval'

print(f'Total crops in index : {collection.count()}')
print(f'Export root          : {export_root}')

## Background sample (run once)

Loads a sample of crop embeddings from the full index. Used as background context in the PCA scatter — shows where retrieved crops sit relative to the whole index.

In [ ]:
# Sample the full index once to provide background context for the PCA scatter.
# No need to re-run this every time you change the question.
BACKGROUND_N = 600

print(f'Sampling {BACKGROUND_N} crops from the index ...')
bg_result = collection.get(limit=BACKGROUND_N, include=['embeddings', 'metadatas'])
bg_embeds  = np.array(bg_result['embeddings'])   # (N, dim)
bg_metas   = bg_result['metadatas']
print(f'Background ready: {bg_embeds.shape}')

## Query

**Change `QUESTION`, `TOP_K`, and `CROP_LABEL` here, then re-run this cell.**

Results are exported to `tmp/retrieval/<sanitized_question>/` and printed as a ranked list.

In [ ]:
# ── Parameters ─────────────────────────────────────────────────────────────
QUESTION   = 'What are the identified sources of errors ?'

TOP_K      = 50       # number of crops to retrieve

# Filter by crop type — set to None for no filter.
# Common values: 'text', 'figure', 'table', 'overview'
CROP_LABEL = None
# ───────────────────────────────────────────────────────────────────────────

print(f'Question   : {QUESTION}')
print(f'Top-K      : {TOP_K}')
print(f'Crop label : {CROP_LABEL or "(all)"}')
print()

# --- Embed the query ---
print('Encoding query ...')
query_embedding = embed_text(QUESTION)
print(f'Embedding dim: {len(query_embedding)}')

# --- Query ChromaDB ---
where_filter = {'crop_label': CROP_LABEL} if CROP_LABEL else None
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=TOP_K,
    where=where_filter,
    include=['metadatas', 'distances', 'embeddings'],
)

ids        = results['ids'][0]
distances  = results['distances'][0]
metadatas  = results['metadatas'][0]
hit_embeds = np.array(results['embeddings'][0])  # stored embeddings of retrieved crops

print(f'Retrieved {len(ids)} crops\n')
print(f'{"Rank":<5} {"Dist":>6}  {"Page":>4}  {"Label":<12}  {"Source"}')
print('─' * 80)
for rank, (crop_id, dist, meta) in enumerate(zip(ids, distances, metadatas), 1):
    source  = Path(meta.get('source', '')).name
    page    = meta.get('page_number', '?')
    label   = meta.get('crop_label') or 'unknown'
    print(f'{rank:<5} {dist:>6.4f}  {str(page):>4}  {label:<12}  {source}')

# --- Export crops ---
safe_name  = ''.join(c if c.isalnum() or c in ' _-' else '_' for c in QUESTION)[:60].strip()
export_dir = export_root / safe_name
export_dir.mkdir(parents=True, exist_ok=True)

exported = 0
print(f'\nExporting to {export_dir} ...')
for rank, (crop_id, dist, meta) in enumerate(zip(ids, distances, metadatas), 1):
    source  = Path(meta.get('source', 'unknown')).stem
    page    = meta.get('page_number', 0)
    label   = meta.get('crop_label') or 'unknown'
    fname   = f'{rank:02d}_dist{dist:.4f}_p{page}_{label}_{source[:40]}.png'
    try:
        img = kvstore.retrieve_image(crop_id)
        img.load()
        img.save(export_dir / fname, format='PNG')
        exported += 1
    except KeyError:
        print(f'  [missing in kvstore] rank={rank} id={crop_id}')

print(f'Exported {exported}/{len(ids)} crops → {export_dir}')

## HyDE — Hypothetical Document Embeddings

**The idea:** instead of embedding the *question*, have the LLM write a short *hypothetical document passage* that would answer it, then embed *that*.

A raw question is linguistically very different from how the answer appears in a document. The embedding model was trained to align document crops with document-like text, so a hypothetical passage lands in a much better region of the embedding space — even if its specific values are wrong.

**How it works here:**
1. A text-only colette service (no RAG) is created using the same LLM from `config.json`
2. The LLM generates a short hypothetical passage in response to the question
3. That passage is embedded (via `embed_text`) and used for retrieval instead of the raw question
4. Results are compared side by side

Run the *Init LLM service* cell once. Then re-run *Generate passage* + the two comparison cells each time you change `QUESTION`.

In [ ]:
# ── Init LLM service for HyDE generation (run once) ─────────────────────────
# Creates a text-only colette service — no RAG, just the LLM.
# Uses a SEPARATE app directory from the main index: llmservice.py always opens
# kvstore.db in append (read-write) mode, and HDF5 forbids the same file to be
# open read-only (this notebook) and read-write (the service) simultaneously.
# A fresh directory gets its own empty kvstore that is never accessed (no RAG).

import asyncio
import shutil
from colette.jsonapi import JSONApi
from colette.apidata import APIData

HYDE_SVC     = 'retrieval_debugger_hyde'
hyde_app_dir = colette_root / 'tmp' / 'hyde_svc'

# Wipe and recreate to guarantee a clean kvstore on every run
if hyde_app_dir.exists():
    shutil.rmtree(hyde_app_dir)
hyde_app_dir.mkdir(parents=True)

llm_cfg = cfg['parameters']['llm']

hyde_init_config = {
    'app': {
        'repository':        str(hyde_app_dir),   # separate dir → own empty kvstore
        'create_repository': True,
        'models_repository': str(models_repo),
        'verbose':           'warning',
        'log_in_app_dir':    False,
    },
    'parameters': {
        'input': {
            'lib': 'hf',
            # no 'rag' → ragobj = None → no retrieval, LLM receives empty docs
        },
        'llm': {
            'source':           llm_cfg['source'],
            'inference':        llm_cfg['inference'],
            'shared_model':     True,
            'gpu_ids':          llm_cfg.get('gpu_ids', [1]),
            'context_size':     1024,
            'disable_thinking': True,
            'system_prompt':    'You are a helpful technical assistant.',
            'dtype':            llm_cfg.get('dtype', 'auto'),
        },
        'output': {
            'num_tokens':    150,
            'base64':        False,
            'output_format': 'json',
        },
    },
}

colette_api_hyde = JSONApi()

try:
    asyncio.run(colette_api_hyde.service_delete(HYDE_SVC))  # service_delete is async
except Exception:
    pass

colette_api_hyde.service_create(HYDE_SVC, APIData(**hyde_init_config))
print(f'LLM service "{HYDE_SVC}" ready  (app dir: {hyde_app_dir})')

In [ ]:
# ── Generate hypothetical passage (re-run when QUESTION changes) ─────────────
HYDE_PROMPT = (
    "Write a short passage from a technical datasheet that directly answers "
    "the following question. Include specific numerical values and units where relevant. "
    "Write only the passage — no preamble, no question restatement.\n\n"
    f"Question: {QUESTION}\n\nPassage:"
)

print("Calling LLM for hypothetical passage ...")
hyde_response = colette_api_hyde.service_predict(
    HYDE_SVC,
    APIData(**{'parameters': {'input': {'message': HYDE_PROMPT}}}),
)
HYDE_PASSAGE = (hyde_response.output or '').strip()

print(f"\nHypothetical passage ({len(HYDE_PASSAGE.split())} words):\n{'─'*70}")
print(HYDE_PASSAGE)
print('─' * 70)

In [ ]:
# ── Step 2: embed the hypothetical passage and retrieve ─────────────────────
print("Embedding hypothetical passage ...")
hyde_embedding = embed_text(HYDE_PASSAGE)

where_filter = {'crop_label': CROP_LABEL} if CROP_LABEL else None
hyde_results = collection.query(
    query_embeddings=[hyde_embedding],
    n_results=TOP_K,
    where=where_filter,
    include=['metadatas', 'distances', 'embeddings'],
)

hyde_ids       = hyde_results['ids'][0]
hyde_distances = hyde_results['distances'][0]
hyde_metadatas = hyde_results['metadatas'][0]
hyde_embeds    = np.array(hyde_results['embeddings'][0])

# ── Step 3: side-by-side comparison ─────────────────────────────────────────
print(f'\n{"Rank":<5}  {"── Original query ──":<42}  {"── HyDE passage ──"}')
print(f'{"":5}  {"Dist":>6}  {"Label":<12}  {"Source":<20}  {"Dist":>6}  {"Label":<12}  {"Source"}')
print('─' * 108)

for rank in range(min(TOP_K, len(ids), len(hyde_ids))):
    # Original
    o_src   = Path(metadatas[rank].get('source', '')).name[:22] if rank < len(metadatas) else ''
    o_dist  = distances[rank]  if rank < len(distances)  else 0.0
    o_label = (metadatas[rank].get('crop_label') or 'unknown')[:10] if rank < len(metadatas) else ''
    # HyDE
    h_src   = Path(hyde_metadatas[rank].get('source', '')).name[:22] if rank < len(hyde_metadatas) else ''
    h_dist  = hyde_distances[rank] if rank < len(hyde_distances) else 0.0
    h_label = (hyde_metadatas[rank].get('crop_label') or 'unknown')[:10] if rank < len(hyde_metadatas) else ''

    changed = '←' if o_src != h_src else ' '
    print(f'{rank+1:<5}  {o_dist:>6.4f}  {o_label:<12}  {o_src:<22}  {h_dist:>6.4f}  {h_label:<12}  {h_src}  {changed}')

# ── Step 4: expected source comparison ──────────────────────────────────────
EXPECTED_SOURCE = 'COLETTEv2_Restitution_2025_03_07_v0.3_JB_light.pdf'   # ← change to the PDF you expect to answer the question

def first_rank_in(id_list, meta_list, expected):
    for i, m in enumerate(meta_list):
        if Path(m.get('source', '')).name == expected:
            return i + 1
    return None

orig_rank = first_rank_in(ids, metadatas, EXPECTED_SOURCE)
hyde_rank = first_rank_in(hyde_ids, hyde_metadatas, EXPECTED_SOURCE)

print(f'\n{"─"*60}')
print(f'Expected source : {EXPECTED_SOURCE}')
print(f'  Original query  → first appears at rank #{orig_rank or ">top-K"}')
print(f'  HyDE passage    → first appears at rank #{hyde_rank or ">top-K"}')
if orig_rank and hyde_rank:
    delta = orig_rank - hyde_rank
    if delta > 0:
        print(f'  Improvement     : +{delta} positions')
    elif delta < 0:
        print(f'  Regression      : {delta} positions')
    else:
        print(f'  No change in rank of expected source')

In [ ]:
# ── HyDE vs original crop grids (top 8, side by side) ───────────────────────
SHOW_N = min(8, TOP_K)

fig, axes = plt.subplots(2, SHOW_N, figsize=(SHOW_N * 3.5, 8))
titles = ['Original query', 'HyDE passage']

for row_idx, (row_ids, row_dists, row_metas) in enumerate([
    (ids[:SHOW_N], distances[:SHOW_N], metadatas[:SHOW_N]),
    (hyde_ids[:SHOW_N], hyde_distances[:SHOW_N], hyde_metadatas[:SHOW_N]),
]):
    for col_idx, (crop_id, dist, meta) in enumerate(zip(row_ids, row_dists, row_metas)):
        ax    = axes[row_idx][col_idx]
        label = meta.get('crop_label') or 'unknown'
        page  = meta.get('page_number', '?')
        src   = Path(meta.get('source', '')).name
        color = LABEL_COLORS.get(label, '#999999')
        # highlight crops from the expected source in green
        border_color = '#00aa00' if src == EXPECTED_SOURCE else color
        border_w     = 5         if src == EXPECTED_SOURCE else 2

        try:
            img = kvstore.retrieve_image(crop_id)
            img.load()
            ax.imshow(np.array(img.convert('RGB')))
        except KeyError:
            ax.set_facecolor('#f0f0f0')
            ax.text(0.5, 0.5, 'NOT FOUND', ha='center', va='center',
                    transform=ax.transAxes, color='red', fontsize=8)

        ax.set_title(
            f'#{col_idx+1}  d={dist:.4f}\n{label} · p{page}\n{src[:28]}',
            fontsize=7,
            color=border_color,
            pad=3,
        )
        for spine in ax.spines.values():
            spine.set_edgecolor(border_color)
            spine.set_linewidth(border_w)
        ax.set_xticks([])
        ax.set_yticks([])

    axes[row_idx][0].set_ylabel(titles[row_idx], fontsize=10, labelpad=6)

plt.suptitle(
    f'Top-{SHOW_N} crop comparison\nGreen border = expected source ({EXPECTED_SOURCE})',
    fontsize=11, y=1.01,
)
plt.tight_layout()
plt.show()

## Retrieval Diagnosis

**The product-identity problem:** The embedding model encodes *semantics* (what kind of content), not *identity* (which specific document or product). Two pages about the same topic but from different documents look nearly identical in embedding space — same parameter names, same layout, near-zero cosine distance.

Rare identifier strings (product codes, part numbers, serial numbers) in the question are often not meaningful to the embedding model since it has no product-specific training. The dominant signal is the *topic* (e.g. `[isolation] [capacitance] [table]`), which can match many documents equally.

**Why the service does better in hybrid mode:** The app can run in `hybrid` mode — BM25 (Tantivy) + embedding in parallel. Tantivy finds the exact identifier string in page text and scores it high lexically. The notebook does pure embedding search, skipping that.

**Two diagnostic cells below:**
1. *Source distribution* — where does the expected PDF first appear and how spread are results across the index?
2. *Query variant comparison* — does rephrasing help? (e.g. putting the identifier first, changing wording)

In [ ]:
# ── Source distribution + expected source analysis ──────────────────────────
# Where does the expected PDF actually rank? How spread are the top results?
# Set EXPECTED_SOURCE and SEARCH_DEPTH below, then run.

EXPECTED_SOURCE = 'COLETTEv2_Restitution_2025_03_07_v0.3_JB_light.pdf'   # ← the PDF you expect to answer the question
SEARCH_DEPTH    = 100                    # how far to look

# Extended retrieval to find expected source
ext = collection.query(
    query_embeddings=[query_embedding],
    n_results=SEARCH_DEPTH,
    include=['metadatas', 'distances'],
)
ext_dists = ext['distances'][0]
ext_metas = ext['metadatas'][0]

top1_dist = ext_dists[0]
top1_name = Path(ext_metas[0].get('source', '')).name

# Source distribution
source_ranks = {}
for i, (dist, meta) in enumerate(zip(ext_dists, ext_metas)):
    name = Path(meta.get('source', '')).name
    if name not in source_ranks:
        source_ranks[name] = {'first_rank': i + 1, 'count': 0, 'best_dist': dist}
    source_ranks[name]['count'] += 1

sorted_sources = sorted(source_ranks.items(), key=lambda x: x[1]['first_rank'])

print(f'Top-{SEARCH_DEPTH} source distribution:\n')
print(f'{"PDF":<45}  {"First rank":>10}  {"Count":>5}  {"Best dist":>9}')
print('─' * 75)
for name, info in sorted_sources[:20]:
    marker = '  ← EXPECTED' if name == EXPECTED_SOURCE else ''
    print(f'{name:<45}  #{info["first_rank"]:>9}  {info["count"]:>5}  {info["best_dist"]:.4f}{marker}')
if len(sorted_sources) > 20:
    print(f'  ... and {len(sorted_sources) - 20} more PDFs')

# Expected source analysis
exp_hits = [
    (i + 1, d, m)
    for i, (d, m) in enumerate(zip(ext_dists, ext_metas))
    if Path(m.get('source', '')).name == EXPECTED_SOURCE
]

print(f'\n{"─"*75}')
print(f'Expected source: {EXPECTED_SOURCE}')
if exp_hits:
    best_rank, best_dist, best_meta = exp_hits[0]
    gap = best_dist - top1_dist
    print(f'  Best rank   : #{best_rank}  (out of {SEARCH_DEPTH} searched)')
    print(f'  Best dist   : {best_dist:.4f}  (top-1 is {top1_dist:.4f})')
    print(f'  Distance gap: +{gap:.4f}  — the expected source is {gap/top1_dist*100:.1f}% further than the winner')
    print(f'  Best crop   : {best_meta.get("crop_label","?")} p{best_meta.get("page_number","?")}')
    print()
    print(f'  All {len(exp_hits)} hits in top-{SEARCH_DEPTH}:')
    for rank, dist, meta in exp_hits:
        print(f'    #{rank:>4}  d={dist:.4f}  {meta.get("crop_label","?"):<12} p{meta.get("page_number","?")}')
else:
    print(f'  Not found in top-{SEARCH_DEPTH}!')

# Bar chart: first-rank-per-source for top sources
fig, ax = plt.subplots(figsize=(10, 5))
top_sources  = sorted_sources[:15]
source_names = [s[0][:38] for s in top_sources]
first_ranks  = [s[1]['first_rank'] for s in top_sources]
counts       = [s[1]['count'] for s in top_sources]
bar_colors   = ['#C44E52' if n == EXPECTED_SOURCE else '#4C72B0' for n, _ in top_sources]

bars = ax.barh(range(len(top_sources)), first_ranks, color=bar_colors, height=0.6, alpha=0.8)
ax.set_yticks(range(len(top_sources)))
ax.set_yticklabels(source_names, fontsize=8)
ax.invert_yaxis()
ax.set_xlabel('First appearance (rank) — lower = appears earlier')
ax.set_title(f'When does each PDF first appear in top-{SEARCH_DEPTH} results?\n'
             f'(red = expected source, question: "{QUESTION[:55]}...")')
for bar, rank, cnt in zip(bars, first_ranks, counts):
    ax.text(rank + 0.3, bar.get_y() + bar.get_height() / 2,
            f'#{rank}  ({cnt} crops)', va='center', ha='left', fontsize=7)
plt.tight_layout()
plt.show()

In [ ]:
# ── Query variant comparison ─────────────────────────────────────────────────
# Shows how rephrasing the question affects which PDF ranks first and where
# the expected source lands. Run after the Query cell.

EXPECTED_SOURCE = 'COLETTEv2_Restitution_2025_03_07_v0.3_JB_light.pdf'   # ← change to the PDF you expect to answer

QUERY_VARIANTS = [
    ('Original',      QUESTION),
    ('Keywords first', 'keyword1 keyword2 specification'),    # ← put key terms first
    ('With context',   'specification table keyword1 keyword2'),
    ('Explicit',       'What is the value of <parameter> in <product>?'),
]

SEARCH_DEPTH = 100   # how deep to look for the expected source

rows = []
for variant_label, q in QUERY_VARIANTS:
    emb = embed_text(q)
    r = collection.query(
        query_embeddings=[emb],
        n_results=SEARCH_DEPTH,
        include=['metadatas', 'distances'],
    )
    vdists = r['distances'][0]
    vmetas = r['metadatas'][0]

    top1_name = Path(vmetas[0].get('source', '')).name
    top1_dist = vdists[0]

    exp_hits = [
        (i + 1, d, m)
        for i, (d, m) in enumerate(zip(vdists, vmetas))
        if Path(m.get('source', '')).name == EXPECTED_SOURCE
    ]
    best_rank = exp_hits[0][0] if exp_hits else None
    best_dist = exp_hits[0][1] if exp_hits else None
    best_label = exp_hits[0][2].get('crop_label', '?') if exp_hits else '—'
    best_page  = exp_hits[0][2].get('page_number', '?') if exp_hits else '—'

    rows.append({
        'variant': variant_label,
        'query':   q,
        'top1_name': top1_name,
        'top1_dist': top1_dist,
        'exp_rank':  best_rank,
        'exp_dist':  best_dist,
        'exp_label': best_label,
        'exp_page':  best_page,
    })
    print(f'[{variant_label}] top-1 → {top1_name}  (d={top1_dist:.4f})')

print()
print(f'Expected source: {EXPECTED_SOURCE}')
print()
print(f'{"Variant":<18}  {"Top-1 PDF":<38}  {"d_top1":>6}  '
      f'{"Exp rank":>8}  {"d_exp":>7}  {"Δ dist":>7}  {"Exp label/page"}')
print('─' * 105)
for row in rows:
    delta = (row['exp_dist'] - row['top1_dist']) if row['exp_dist'] is not None else float('nan')
    exp_rank_str = f"#{row['exp_rank']}" if row['exp_rank'] else f'>top{SEARCH_DEPTH}'
    exp_dist_str = f"{row['exp_dist']:.4f}" if row['exp_dist'] is not None else '  —   '
    delta_str    = f"+{delta:.4f}" if not (delta != delta) else '  —   '
    loc_str      = f"{row['exp_label']} p{row['exp_page']}" if row['exp_rank'] else '—'
    print(f"{row['variant']:<18}  {row['top1_name']:<38}  {row['top1_dist']:.4f}  "
          f"{exp_rank_str:>8}  {exp_dist_str:>7}  {delta_str:>7}  {loc_str}")

# Plot: distance comparison across variants
fig, ax = plt.subplots(figsize=(10, 4))
x = range(len(rows))
top1_dists = [r['top1_dist'] for r in rows]
exp_dists  = [r['exp_dist'] if r['exp_dist'] is not None else float('nan') for r in rows]
variant_labels = [r['variant'] for r in rows]

ax.plot(x, top1_dists, 'o--', color='#C44E52', label='top-1 hit (any source)', linewidth=1.5)
ax.plot(x, exp_dists,  's-',  color='#55A868', label=f'best hit from {EXPECTED_SOURCE}', linewidth=1.5)
ax.fill_between(x, top1_dists, exp_dists, alpha=0.1, color='gray', label='gap')
ax.set_xticks(list(x))
ax.set_xticklabels(variant_labels, rotation=12)
ax.set_ylabel('Cosine distance  (lower = closer to query)')
ax.set_title(f'Query rephrasing — does it bring "{EXPECTED_SOURCE}" closer?')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## Visualize results

Re-run the two cells below after each query (model output is already computed — these are fast).

**Cell 1 — Crop grid:** shows every retrieved crop as an image, ranked by distance.  Border colour = crop type (blue=text, green=figure, red=table, purple=overview).

**Cell 2 — Embedding space + Distance profile:**
- *PCA scatter* — projects the full index (grey), retrieved crops (coloured) and your query (★) into 2D. Dotted lines connect query to each result. Hover for metadata.  Note: PCA compresses 1536 dims → 2, so spatial closeness is approximate, but cluster membership is real.
- *Distance bar chart* — exact cosine distances per result. The gap between the first and second bar tells you how decisive the top hit was.

In [ ]:
# ── Crop thumbnail grid ─────────────────────────────────────────────────────
N_COLS = 4
n      = len(ids)
n_rows = (n + N_COLS - 1) // N_COLS

fig, axes = plt.subplots(n_rows, N_COLS, figsize=(N_COLS * 4, n_rows * 4.2))
axes = np.array(axes).reshape(-1)

for idx, (crop_id, dist, meta) in enumerate(zip(ids, distances, metadatas)):
    ax     = axes[idx]
    label  = meta.get('crop_label') or 'unknown'
    page   = meta.get('page_number', '?')
    source = Path(meta.get('source', '')).name
    color  = LABEL_COLORS.get(label, '#999999')

    try:
        img = kvstore.retrieve_image(crop_id)
        img.load()                              # force full decode before buffer goes out of scope
        ax.imshow(np.array(img.convert('RGB')))
    except KeyError:
        ax.set_facecolor('#f0f0f0')
        ax.text(0.5, 0.5, 'NOT FOUND', ha='center', va='center',
                transform=ax.transAxes, color='red', fontsize=10)

    ax.set_title(
        f'#{idx+1}  d={dist:.4f}\n{label} · p{page}\n{source[:35]}',
        fontsize=8, color=color, pad=4,
    )
    for spine in ax.spines.values():
        spine.set_edgecolor(color)
        spine.set_linewidth(3)
    ax.set_xticks([])
    ax.set_yticks([])

for idx in range(n, len(axes)):
    axes[idx].set_visible(False)

q_short = QUESTION if len(QUESTION) <= 72 else QUESTION[:69] + '...'
plt.suptitle(f'Retrieved crops for:\n"{q_short}"', fontsize=11, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Two complementary views of the retrieval results
#   Plot 1 — Document proximity  : TRUE cosine distances, query at centre
#   Plot 2 — Crop-type space     : PCA semantic embedding (distances approximate)
# ══════════════════════════════════════════════════════════════════════════════

# ── Shared setup ─────────────────────────────────────────────────────────────
q_arr   = np.array(query_embedding).reshape(1, -1)
all_emb = np.vstack([bg_embeds, hit_embeds, q_arr])

pca  = PCA(n_components=2)
proj = pca.fit_transform(all_emb)

n_bg       = len(bg_embeds)
n_hit      = len(hit_embeds)
bg_proj    = proj[:n_bg]
hit_proj   = proj[n_bg:n_bg + n_hit]
q_proj     = proj[-1:]
var1, var2 = pca.explained_variance_ratio_

# Open-outline symbols — one per document
PLOTLY_SYMBOLS = [
    'circle-open', 'square-open', 'diamond-open', 'cross-open', 'x-open',
    'triangle-up-open', 'triangle-down-open', 'triangle-left-open', 'triangle-right-open',
    'pentagon-open', 'hexagon-open', 'star-open', 'hexagram-open',
    'star-triangle-up-open', 'star-square-open', 'star-diamond-open',
    'circle-open-dot', 'square-open-dot', 'diamond-open-dot',
    'triangle-up-open-dot', 'triangle-down-open-dot',
]
# Sort sources by their best (smallest) cosine distance — closest document first
src_names = list(dict.fromkeys(Path(m.get('source', '')).name for m in metadatas))
src_best  = {
    src: min(distances[i] for i, m in enumerate(metadatas)
             if Path(m.get('source', '')).name == src)
    for src in src_names
}
unique_sources = sorted(src_names, key=lambda s: src_best[s])  # closest → farthest
src_symbol     = {src: PLOTLY_SYMBOLS[i % len(PLOTLY_SYMBOLS)]
                  for i, src in enumerate(unique_sources)}

present_labels = {m.get('crop_label') or 'unknown' for m in metadatas}

# Dynamic figure height so legend never overflows
n_legend   = len(unique_sources) + len(present_labels) + 3
fig_height = max(720, n_legend * 24 + 120)


# ══════════════════════════════════════════════════════════════════════════════
# PLOT 1 — Document proximity  (TRUE cosine distances)
# Each document occupies an angular sector sorted by best distance.
# Radius from centre = actual cosine distance → no PCA distortion.
# ══════════════════════════════════════════════════════════════════════════════

n_src   = len(unique_sources)
sector  = 2 * np.pi / n_src      # radians per document sector

# Assign each crop an angle within its document's sector
src_crop_counts = {src: [] for src in unique_sources}
for i, m in enumerate(metadatas):
    src_crop_counts[Path(m.get('source', '')).name].append(i)

angles_rad = np.zeros(len(metadatas))
for src_idx, src in enumerate(unique_sources):
    idxs   = src_crop_counts[src]
    center = src_idx * sector + sector / 2   # centre of this document's sector
    n      = len(idxs)
    spread = sector * 0.75                   # use 75% of the sector width
    for k, crop_i in enumerate(idxs):
        if n == 1:
            angles_rad[crop_i] = center
        else:
            angles_rad[crop_i] = center + (k / (n - 1) - 0.5) * spread

radii = np.array(distances)
x_rad = radii * np.cos(angles_rad)
y_rad = radii * np.sin(angles_rad)

fig1 = go.Figure()

# Reference distance rings
d_min, d_max, d_mean = min(radii), max(radii), radii.mean()
theta_ring = np.linspace(0, 2 * np.pi, 300)
for r, lbl, dash, opacity in [
    (d_min,  f'closest  {d_min:.4f}',  'dot',  0.5),
    (d_mean, f'mean  {d_mean:.4f}',    'dash', 0.5),
    (d_max,  f'farthest  {d_max:.4f}', 'dot',  0.5),
]:
    fig1.add_trace(go.Scatter(
        x=r * np.cos(theta_ring), y=r * np.sin(theta_ring),
        mode='lines',
        line=dict(color=f'rgba(150,150,150,{opacity})', width=1, dash=dash),
        name=lbl,
        legendgroup='rings',
        legendgrouptitle=dict(text='Distance rings'),
        hoverinfo='skip',
    ))

# Radial spokes marking each document's sector centre (faint)
for src_idx, src in enumerate(unique_sources):
    center = src_idx * sector + sector / 2
    r_max  = d_max * 1.05
    fig1.add_trace(go.Scatter(
        x=[0, r_max * np.cos(center)],
        y=[0, r_max * np.sin(center)],
        mode='lines',
        line=dict(color='rgba(200,200,200,0.35)', width=1),
        showlegend=False,
        hoverinfo='skip',
    ))
    # Label at rim
    fig1.add_trace(go.Scatter(
        x=[r_max * 1.06 * np.cos(center)],
        y=[r_max * 1.06 * np.sin(center)],
        mode='text',
        text=[src[:28]],
        textfont=dict(size=8, color='#555'),
        showlegend=False,
        hoverinfo='skip',
    ))

# Lines from query to each crop
for i in range(len(distances)):
    fig1.add_trace(go.Scatter(
        x=[0, x_rad[i]], y=[0, y_rad[i]],
        mode='lines',
        line=dict(color='rgba(180,180,180,0.25)', width=0.8),
        showlegend=False, hoverinfo='skip',
    ))

# Retrieved crops — one trace per source (open symbols, colour = crop type)
for src in unique_sources:
    idxs = src_crop_counts[src]
    fig1.add_trace(go.Scatter(
        x=[x_rad[i] for i in idxs],
        y=[y_rad[i] for i in idxs],
        mode='markers+text',
        marker=dict(
            size=18,
            symbol=src_symbol[src],
            color=[LABEL_COLORS.get(metadatas[i].get('crop_label') or 'unknown', '#999') for i in idxs],
            line=dict(
                width=2.5,
                color=[LABEL_COLORS.get(metadatas[i].get('crop_label') or 'unknown', '#999') for i in idxs],
            ),
        ),
        text=[f'#{i+1}' for i in idxs],
        textposition='top center',
        textfont=dict(size=8, color='#222'),
        hovertext=[
            f"<b>#{i+1}</b>  dist={distances[i]:.4f}<br>"
            f"{metadatas[i].get('crop_label','?')} · p{metadatas[i].get('page_number','?')}<br>"
            f"{Path(metadatas[i].get('source','')).name}"
            for i in idxs
        ],
        hovertemplate='%{hovertext}<extra></extra>',
        name=f'{src[:38]}  (best {src_best[src]:.4f})',
        legendgroup='sources',
        legendgrouptitle=dict(text='Document → shape  (sorted closest first)'),
    ))

# Crop-type colour legend (dummy)
for lbl, color in LABEL_COLORS.items():
    if lbl in present_labels:
        fig1.add_trace(go.Scatter(
            x=[None], y=[None], mode='markers',
            marker=dict(size=13, color=color, symbol='circle', line=dict(width=1.5, color='#333')),
            name=lbl, legendgroup='labels',
            legendgrouptitle=dict(text='Crop type → colour'),
            showlegend=True,
        ))

# Query at centre
fig1.add_trace(go.Scatter(
    x=[0], y=[0], mode='markers',
    marker=dict(size=22, symbol='star', color='red', line=dict(width=2, color='darkred')),
    hovertemplate='<b>Query ★</b><br>centre — distance = 0<extra></extra>',
    name='query ★', legendgroup='other', legendgrouptitle=dict(text='Other'),
))

fig1.update_layout(
    title=dict(
        text=(
            'Document proximity — TRUE cosine distances<br>'
            '<sup>★ query at centre · <b>radius = actual cosine distance</b> · '
            'sector = document (sorted closest → farthest) · colour = crop type</sup>'
        ),
        x=0.5,
    ),
    xaxis=dict(scaleanchor='y', scaleratio=1, showgrid=False, zeroline=False,
               showticklabels=False, title=''),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False, title=''),
    height=fig_height,
    legend=dict(x=1.01, xanchor='left', y=1, yanchor='top',
                groupclick='toggleitem', tracegroupgap=12, font=dict(size=11)),
    margin=dict(r=340, t=80, b=20, l=20),
    plot_bgcolor='white',
)
fig1.show()


# ══════════════════════════════════════════════════════════════════════════════
# PLOT 2 — Crop-type view  (PCA semantic space — distances are approximate)
# ══════════════════════════════════════════════════════════════════════════════

fig2 = go.Figure()

# Background
fig2.add_trace(go.Scatter(
    x=bg_proj[:, 0], y=bg_proj[:, 1], mode='markers',
    marker=dict(size=4, color='#cccccc', opacity=0.40),
    hovertext=[
        f"{m.get('crop_label','?')} · p{m.get('page_number','?')} · {Path(m.get('source','')).name}"
        for m in bg_metas
    ],
    hovertemplate='%{hovertext}<extra>index sample</extra>',
    name='index sample', legendgroup='misc', legendgrouptitle=dict(text='Other'),
))

# Dotted lines query → crops
for i in range(n_hit):
    fig2.add_trace(go.Scatter(
        x=[q_proj[0, 0], hit_proj[i, 0]],
        y=[q_proj[0, 1], hit_proj[i, 1]],
        mode='lines',
        line=dict(color='rgba(200,0,0,0.18)', width=1, dash='dot'),
        showlegend=False, hoverinfo='skip',
    ))

# One trace per crop type (filled circle, colour = type)
for lbl, color in LABEL_COLORS.items():
    idxs = [i for i, m in enumerate(metadatas) if (m.get('crop_label') or 'unknown') == lbl]
    if not idxs:
        continue
    fig2.add_trace(go.Scatter(
        x=[hit_proj[i, 0] for i in idxs],
        y=[hit_proj[i, 1] for i in idxs],
        mode='markers+text',
        marker=dict(size=16, symbol='circle', color=color,
                    line=dict(width=1.5, color='black')),
        text=[f'#{i+1}' for i in idxs],
        textposition='top center',
        textfont=dict(size=9, color='black'),
        hovertext=[
            f"<b>#{i+1}</b>  dist={distances[i]:.4f}<br>"
            f"{lbl} · p{metadatas[i].get('page_number','?')}<br>"
            f"{Path(metadatas[i].get('source','')).name}"
            for i in idxs
        ],
        hovertemplate='%{hovertext}<extra></extra>',
        name=lbl, legendgroup='labels',
        legendgrouptitle=dict(text='Crop type → colour'),
    ))

# Query star
fig2.add_trace(go.Scatter(
    x=q_proj[:, 0], y=q_proj[:, 1], mode='markers',
    marker=dict(size=22, symbol='star', color='red', line=dict(width=2, color='darkred')),
    hovertemplate=f'<b>Query ★</b><br>{QUESTION}<extra></extra>',
    name='query ★', legendgroup='misc',
))

fig2.update_layout(
    title=dict(
        text=(
            f'Crop-type semantic space — PCA 2D  ({(var1+var2):.1%} variance explained)<br>'
            f'<sup>★ query  |  colour = crop type  |  · index sample  '
            f'|  ⚠ PCA position ≈ embedding direction, NOT true cosine distance — use Plot 1 for distances</sup>'
        ),
        x=0.5,
    ),
    xaxis_title=f'PC1 ({var1:.1%})',
    yaxis_title=f'PC2 ({var2:.1%})',
    height=600,
    legend=dict(x=1.01, xanchor='left', y=1, yanchor='top',
                groupclick='toggleitem', tracegroupgap=14, font=dict(size=11)),
    margin=dict(r=200),
)
fig2.show()

# ── Distance bar chart ────────────────────────────────────────────────────────
labels_y   = [
    f"#{i+1}  {m.get('crop_label') or '?'} · p{m.get('page_number','?')} · {Path(m.get('source','')).name[:32]}"
    for i, m in enumerate(metadatas)
]
bar_colors = [LABEL_COLORS.get(m.get('crop_label') or 'unknown', '#999') for m in metadatas]

fig3, ax = plt.subplots(figsize=(9, max(3, len(ids) * 0.7)))
bars = ax.barh(range(len(ids)), distances, color=bar_colors, edgecolor='white', height=0.6)
ax.set_yticks(range(len(ids)))
ax.set_yticklabels(labels_y, fontsize=9)
ax.invert_yaxis()
ax.set_xlabel('Cosine distance  (lower = more similar to query)')
q_short = QUESTION if len(QUESTION) <= 55 else QUESTION[:52] + '...'
ax.set_title(f'Distance profile — "{q_short}"')

ax.axvline(x=d_mean, color='gray', linestyle='--', alpha=0.7, label=f'mean = {d_mean:.4f}')

for bar, dist in zip(bars, distances):
    ax.text(dist + 0.0005, bar.get_y() + bar.get_height() / 2,
            f'{dist:.4f}', va='center', ha='left', fontsize=8)

legend_patches = [
    mpatches.Patch(color=LABEL_COLORS.get(lbl, '#999'), label=lbl)
    for lbl in LABEL_COLORS if lbl in present_labels
] + [mpatches.Patch(color='none', label=f'── mean = {d_mean:.4f}')]
ax.legend(handles=legend_patches, loc='lower right', fontsize=8,
          title='Crop type → colour', title_fontsize=8)

plt.tight_layout()
plt.show()

## Cleanup

## Text Search (BM25 / Tantivy) Debugger

Investigates what the Tantivy BM25 index returns for the current question.

The query sent to Tantivy uses **keyword extraction** (`_extract_keywords`):
- Hyphens replaced by spaces → `PRODUCT-01-A` → tokens `PRODUCT`, `01`, `A` (matching what the indexer stored)
- English and French stop words dropped
- Single-character tokens dropped

The cells below show which pages BM25 retrieves and what text they contain.

In [ ]:
# ── Open the Tantivy text-search index ───────────────────────────────────────
import sys, re as _re, os
sys.path.insert(0, str(colette_root / 'src'))

import importlib
import colette.backends.tantivy.tantivy_index as _ti_mod
importlib.reload(_ti_mod)                       # pick up any edits to the source
from colette.backends.tantivy.tantivy_index import TantivyIndex
import logging as _logging

_ti_logger = _logging.getLogger('tantivy_debug')
text_index_path = app_dir / 'mm_index' / 'text_search_engine'
ti = TantivyIndex(text_index_path, _ti_logger)

if not ti.exists():
    print(f'No text-search index found at {text_index_path}')
    print('Run the indexing step first (or set retrieval_mode=hybrid during indexing).')
else:
    _idx = ti.open()
    _searcher = _idx.searcher()
    print(f'Text index : {text_index_path}')
    print(f'Documents  : {_searcher.num_docs}  pages indexed')

    def _run_query(qstr, limit=8):
        try:
            tq = _idx.parse_query(qstr, ['content', 'source'])
            r  = _searcher.search(tq, limit)
            hits = []
            for score, addr in r.hits:
                d = _searcher.doc(addr)
                src = d['source'][0] if d['source'] else ''
                pn  = d.get_first('page_number') or '?'
                content = (d['content'][0] if d['content'] else '')[:160]
                hits.append({'score': score, 'source': os.path.basename(src),
                             'page': pn, 'preview': content})
            return hits
        except Exception as e:
            return [{'score': 0, 'source': f'PARSE ERROR: {e}', 'page': '', 'preview': ''}]

    _kw   = ti._extract_keywords(QUESTION)
    hits  = _run_query(_kw)

    print(f'\nQuestion : {QUESTION}')
    print(f'Query    : {_kw}')

    _expected = globals().get('EXPECTED_SOURCE', '')
    print(f'\n{"Rank":<5} {"score":>7}  {"PDF":<45}  {"Page"}')
    print('─' * 70)
    for i, h in enumerate(hits):
        marker = ' ✓' if _expected and h['source'] == _expected else ''
        print(f'{i+1:<5} {h["score"]:>7.3f}  {h["source"][:40]:<45}  {h["page"]}{marker}')

In [ ]:
# ── Text snippet preview — what does the winning page actually contain? ───────
# (Requires the BM25 debugger cell above to have run first)
TOP_N_PREVIEW = 5
print(f'Top-{TOP_N_PREVIEW} text-search results (query: "{_kw}")\n')
print(f'{"Rank":<5} {"Score":>8}  {"PDF":<48}  {"Page":>4}  Preview')
print('─' * 120)
for i, h in enumerate(hits[:TOP_N_PREVIEW], 1):
    preview = h['preview'].replace('\n', ' ')[:100]
    print(f'{i:<5} {h["score"]:>8.3f}  {h["source"]:<48}  {str(h["page"]):>4}  {preview}')

# ── Try custom query (edit freely) ───────────────────────────────────────────
CUSTOM_QUERY = QUESTION           # ← replace with any string to experiment
_custom_kw   = ti._extract_keywords(CUSTOM_QUERY)
print(f'\nCustom query  : {CUSTOM_QUERY}')
print(f'  → keywords  : {_custom_kw}')
custom_hits   = _run_query(_custom_kw)
print(f'\n{"Rank":<5} {"Score":>8}  {"PDF":<48}  {"Page":>4}')
print('─' * 75)
for i, h in enumerate(custom_hits, 1):
    print(f'{i:<5} {h["score"]:>8.3f}  {h["source"]:<48}  {str(h["page"]):>4}')

In [ ]:
kvstore.close()

# Free GPU memory if you are done experimenting
del embed_model
torch.cuda.empty_cache()
print('Done.')